In [1]:
import numpy as np 
import pandas as pd 
from scipy.sparse import csr_matrix 
from implicit.als import AlternatingLeastSquares 
 
np.random.seed(4060) 

c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
users = [f"U{i:02d}" for i in range(1, 13)] 
items = [f"Track_{i:02d}" for i in range(1, 19)] 
event_types = ["play", "repeat_play", "save"] 
 
rows = [] 
for user in users: 
    preferred = np.random.choice(items, size=6, replace=False) 
    for item in preferred: 
        n = np.random.randint(1, 5) 
        for _ in range(n): 
            rows.append({ 
                "user_id": user, 
                "item_id": item, 
                "event_type": np.random.choice(event_types, 
                    p=[0.60, 0.25, 0.15]) 
            }) 
 
events = pd.DataFrame(rows) 
events.head(), events.shape 

(  user_id   item_id   event_type
 0     U01  Track_02  repeat_play
 1     U01  Track_02  repeat_play
 2     U01  Track_02         play
 3     U01  Track_02         save
 4     U01  Track_06  repeat_play,
 (194, 3))

In [3]:
events['event_type'].value_counts() 
events.groupby('user_id').size().describe() 

count    12.000000
mean     16.166667
std       3.511885
min      12.000000
25%      13.750000
50%      15.000000
75%      17.750000
max      23.000000
dtype: float64

What can we infer from a repeat play or a save?

A repeat play indicates sustained interest or high replay value.

A save signals explicit intent to add an item to a personal collection or replay it later.

What can we not infer?

We cannot infer explicit rating values. A track might be played because it's part of an automated playlist, background noise, or started accidentally.

We cannot infer negative sentiment directly—an item without plays is not necessarily disliked.

Which biases might appear?

Position/Recommendation Bias: Users tend to interact more with tracks featured at the top of playlists or home feeds.

Autoplay & Passive Listening Bias: Plays generated automatically without active user choice.

Sparsity & Popularity Bias: Popular tracks receive exponentially more exposure and logged events.

In [4]:
event_weight = { 
    "play": 1.0, 
    "repeat_play": 3.0, 
    "save": 5.0 
} 
 
events['weight'] = events['event_type'].map(event_weight) 
interactions = (events 
    .groupby(['user_id', 'item_id'], as_index=False)['weight'] 
    .sum() 
    .rename(columns={'weight': 'strength'})) 
 
interactions.sort_values('strength', ascending=False).head(10) 

,user_id,item_id,strength
71,U12,Track_12,12.0
69,U12,Track_09,12.0
61,U11,Track_03,12.0
1,U01,Track_02,12.0
14,U03,Track_11,11.0
70,U12,Track_10,11.0
67,U12,Track_05,10.0
54,U10,Track_06,10.0
51,U09,Track_08,10.0
46,U08,Track_14,10.0


Should one save always be worth five plays?
Not necessarily. A user might save an entire album after hearing one song and never listen again, whereas 5 repeated plays demonstrate confirmed listening intent.
How could accidental clicks or autoplay distort strength?Autoplay can artificially inflate play counts without active interest. Accidental clicks generate spurious single-play events.
What different weights would you test first?Testing a non-linear or log-transformed weight scheme (e.g., $1 + \log(\text{play count})$) prevents a high volume of passive plays from swiping away explicit saves.

In [5]:
alpha = 20 
interactions['preference'] = (interactions['strength'] > 0).astype(int) 
interactions['confidence'] = 1 + alpha * interactions['strength'] 
 
interactions[['user_id', 'item_id', 'strength', 
              'preference', 'confidence']].head()

,user_id,item_id,strength,preference,confidence
0,U01,Track_01,2.0,1,41.0
1,U01,Track_02,12.0,1,241.0
2,U01,Track_06,6.0,1,121.0
3,U01,Track_08,3.0,1,61.0
4,U01,Track_11,8.0,1,161.0


How do preference and confidence differ for strength = 1 vs. 12?Preference ($p_{ui}$): Equal to 1 in both cases (both reflect observed interactions).Confidence ($c_{ui}$): For strength = 1, $c_{ui} = 1 + 20(1) = 21$. For strength = 12, $c_{ui} = 1 + 20(12) = 241$. The model weighs the square error loss for strength = 12 far more heavily during optimization.Why should unobserved pairs have low confidence rather than be treated as definite dislikes?An unobserved pair ($p_{ui} = 0$) has $c_{ui} = 1$. It signals zero observed interaction, which could mean the user never saw the track. Low confidence lets the model penalize missing pairs softly rather than forcing them to zero latent similarity.What happens if alpha is extremely large?The model overfits strictly to observed pairs and ignores unobserved pairs, resulting in extreme popularity bias and poor generalization to unseen items.

In [6]:
user_ids = sorted(interactions['user_id'].unique()) 
item_ids = sorted(interactions['item_id'].unique()) 
 
user_to_idx = {u: i for i, u in enumerate(user_ids)} 
item_to_idx = {it: i for i, it in enumerate(item_ids)} 
idx_to_item = {i: it for it, i in item_to_idx.items()} 
 
row = interactions['user_id'].map(user_to_idx) 
col = interactions['item_id'].map(item_to_idx) 
data = interactions['strength'].astype(float) 
 
user_item = csr_matrix( 
    (data, (row, col)), 
    shape=(len(user_ids), len(item_ids)) 
) 
 
print('shape:', user_item.shape) 
print('observed interactions:', user_item.nnz) 
print('density:', user_item.nnz / (user_item.shape[0] * 
user_item.shape[1]))

shape: (12, 18)
observed interactions: 72
density: 0.3333333333333333


Checkpoint 4 — Read the matrixWhat does one row represent? What does one column represent?Row: A single user's vector of interaction strengths across all catalog tracks.Column: A single track's vector of interaction strengths across all users.Why is a sparse matrix more appropriate than a dense table?Most user-item pairs are zero. Compressed Sparse Row (CSR) format stores only nonzero entries $(i, j, v)$, reducing memory overhead from $\mathcal{O}(U \times I)$ to $\mathcal{O}(\text{nnz})$.What does density tell us?Density indicates catalog coverage. Extremely low density ($< 1\%$) highlights severe data sparsity where collaborative filtering must leverage shared latent structures.

In [7]:
model = AlternatingLeastSquares( 
factors=16, 
regularization=0.10, 
alpha=alpha, 
iterations=25, 
random_state=4060 
) 
model.fit(user_item) 
print(model.user_factors.shape, model.item_factors.shape) 

c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\implicit\cpu\als.py:96: RuntimeWarning: OpenBLAS is configured to use 8 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()
100%|██████████| 25/25 [00:00<00:00, 112.55it/s]

(12, 16) (18, 16)


What does factors=16 control?It sets the dimensionality $k=16$ of the latent factor space for representing users and items via vector dot products ($u_i \cdot v_j$).Why might regularization help?Regularization ($\lambda = 0.10$) penalizes high magnitude weights in factor matrices, preventing overfitting on rare items or highly active users.Why use a fixed random seed?Latent factor initialization relies on random distributions; fixing random_state guarantees exact reproducibility across notebook executions.

In [8]:
target_user = 'U01' 
u = user_to_idx[target_user] 
 
item_idx, scores = model.recommend( 
    u, 
    user_item[u], 
    N=5, 
    filter_already_liked_items=True 
) 
 
recommendations = pd.DataFrame({ 
    'item_id': [idx_to_item[i] for i in item_idx], 
    'score': scores 
}) 
 
seen = interactions.loc[ 
    interactions['user_id'].eq(target_user), 
    ['item_id', 'strength'] 
].sort_values('strength', ascending=False) 
 
print('Observed history') 
display(seen.head(10)) 
print('Recommended next') 
display(recommendations) 

Observed history


,item_id,strength
1,Track_02,12.0
4,Track_11,8.0
2,Track_06,6.0
5,Track_13,6.0
3,Track_08,3.0
0,Track_01,2.0


Recommended next


,item_id,score
0,Track_05,0.237527
1,Track_04,0.106779
2,Track_12,0.091971
3,Track_10,0.068343
4,Track_07,0.063375


Do the recommended items look plausible?Yes, ALS recommends items whose latent vectors align closely with the weighted sum of user U01's past interacted item factors.Why do model scores not behave like probabilities?ALS output scores represent unconstrained linear projection dot-products ($\hat{p}_{ui} = u_i^T v_j$). They are non-bounded ranking utilities, not normalized probabilities $[0, 1]$.What business or user constraints require reranking?Diversity: Preventing recommendations from being limited to a single genre.Novelty & Freshness: Penalizing over-exposed catalog items.Business rules: Filtering unlicenced tracks or explicit content according to account settings.

In [10]:
# Experiment: Testing alpha = 5 vs alpha = 40
model_exp = AlternatingLeastSquares(
    factors=16,
    regularization=0.10,
    alpha=40,
    iterations=25,
    random_state=4060
)
model_exp.fit(user_item)

exp_item_idx, exp_scores = model_exp.recommend(
    u, user_item[u], N=5, filter_already_liked_items=True
)

comparison = pd.DataFrame({
    'baseline_item (alpha=20)': recommendations['item_id'],
    'experiment_item (alpha=40)': [idx_to_item[i] for i in exp_item_idx]
})
print(comparison)

100%|██████████| 25/25 [00:00<00:00, 166.47it/s]


  baseline_item (alpha=20) experiment_item (alpha=40)
0                 Track_05                   Track_12
1                 Track_04                   Track_04
2                 Track_12                   Track_10
3                 Track_10                   Track_18
4                 Track_07                   Track_07


Why is a random row split risky for event logs?Random splitting causes temporal leakage—training on future events to predict past behavior. Real evaluations must use time-based splits.When would Recall@K matter more than Precision@K?In discovery or streaming settings where the primary goal is presenting at least one relevant item within a small shortlist $K$.What offline result would still not prove user benefit?High offline Recall/NDCG score does not guarantee user satisfaction if recommendations lack diversity, promote filter bubbles, or push stale recommendations.